# Chapter 15: DSGE Models: The Frontier of Business Cycle Research

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/quarcs-lab/macro102-site/blob/main/notebooks/ch15-dsge-models-the-frontier-of-business-cycle-research.ipynb)

*Data Science and Macroeconomic Models: A Python Companion to Intermediate Macroeconomics* · Carlos Mendez

> **How to use this notebook.** In Colab, click *Runtime → Run all*. After that, every model
> figure has live sliders or buttons: change one and the model is solved again on the spot. On the
> book website the same figures come with one precomputed slider. Each section ends with a short
> **Your turn** task, and you can try it in the empty code cell below the task.

## Learning goals

Research on business cycles today uses **DSGE models**: *dynamic*, *stochastic*, *general equilibrium*
models. This chapter looks at where they came from and solves their most important building block, the
labor market. By the end of it you will be able to:

- explain the letters D, S and GE, and how **real business cycle** models grew into DSGE models;
- read U.S. data on total factor productivity (TFP) growth, and see why the way TFP is measured changes the
  answer to "do TFP shocks cause recessions?";
- solve the labor market of a DSGE model, with labor demand from the marginal product of labor and labor supply
  $L^s = \bar{\ell}\,w/c$, and use it to study a TFP shock, a sales tax, government purchases and good news;
- show how **sticky wages** and **sticky prices** change the effects of monetary policy on employment and the
  real wage;
- say what an **impulse response function** shows and what the estimated Smets–Wouters model reports;
- in Python: write one equilibrium function and reuse it in many figures, reshape data with `melt`, draw a
  scatter in two panels with a regression line, and use period sliders and toggle buttons.

## Figures in this chapter

The figures follow the order of the textbook chapter. The second column shows which textbook
figure each notebook figure replicates. Textbook Figures 15.9 to 15.13 are results of a large estimated
model; they are not recomputed, and a short note in the section named in the last column explains each.

| Figure | Textbook figure | Type | What you can change |
|---|---|---|---|
| Figure 1 | 15.1 U.S. total factor productivity growth | data | — (hover for each year; raw and adjusted TFP compared in a second chart) |
| Figure 2 | 15.2 The labor market in the DSGE model | model | TFP Ā, consumption c, labor-supply scale ℓ̄ |
| Figure 3 | 15.3 A negative TFP shock in the DSGE model | model | TFP after the shock Ā′, sales tax τ, consumption c′ |
| Figure 4 | 15.4 The imposition of a sales tax in the DSGE model | model | sales tax τ, TFP Ā′, consumption c′ |
| Figure 5 | 15.5 A rise in government purchases | model | consumption c′, TFP Ā′, sales tax τ |
| Figure 6 | 15.6 Sticky wages in the DSGE model | model | sticky nominal wage W̄, period, new price level P′ |
| Figure 7 | 15.7 A monetary expansion with sticky wages | model | period, new price level P′, sticky nominal wage W̄ |
| Figure 8 | 15.8 A monetary expansion with sticky prices | model | output demanded Y′, consumption c′ |
| — | 15.9 The response of GDP to a monetary policy shock | not replicable | — (Smets–Wouters model output; see the note at the end of Figure 8) |
| — | 15.10 The dynamic effects of a monetary policy shock | not replicable | — (Smets–Wouters model output; see the note at the end of Figure 8) |
| — | 15.11 The dynamic effects of a TFP shock | not replicable | — (Smets–Wouters model output; see the note at the end of Figure 3) |
| — | 15.12 The dynamic effects of a shock to government purchases | not replicable | — (Smets–Wouters model output; see the note at the end of Figure 5) |
| — | 15.13 The dynamic effects of a financial friction shock | not replicable | — (Smets–Wouters model output; see the note at the end of Figure 8) |
| Figure 9 | 15.14 Good news in the DSGE model: flexible prices | model | consumption after the news c′, flexible or sticky prices |
| Figure 10 | 15.15 Good news in the DSGE model: sticky prices | model | flexible or sticky prices, consumption after the news c′ |

## Setup

We first load the Python libraries we need. You only have to run this cell once.

> **Python note.** `import` loads a *library*, a toolbox written by other people. We give each
> library a short nickname, for example `np` for NumPy, so that the code stays short. Text after
> a `#` is a *comment*: Python ignores it, and it is there for human readers. The `try ... except`
> block runs one Colab-only line and simply skips it anywhere else.

In [1]:
# Libraries used in this chapter
import numpy as np                    # numbers and arrays
import pandas as pd                   # data tables
import plotly.express as px           # quick charts from data tables
import plotly.graph_objects as go     # charts built piece by piece (model figures)
import plotly.io as pio
from ipywidgets import interact, FloatSlider, IntSlider, ToggleButtons

# One light blue/green look for every chart
BLUE, TEAL, GREEN, GRAY = "#16528c", "#008070", "#7fb23a", "#8a96a3"
pio.templates.default = "plotly_white"
pio.templates["plotly_white"].layout.update(width=860, height=480)   # same size in Colab and on the website
px.defaults.color_discrete_sequence = [BLUE, TEAL, GREEN, "#5aa0d8", "#0b4f47"]
WIDE = {"description_width": "initial"}         # show the full slider labels

# Colab only: let Plotly charts appear inside the interactive widgets
try:
    import google.colab
    google.colab.output.enable_custom_widget_manager()
except ImportError:
    pass                                        # not running in Colab: nothing to do

# The data files of this chapter live in the book's GitHub repository
DATA_URL = "https://raw.githubusercontent.com/quarcs-lab/macro102-site/main/data/ch15/"

The next cell defines a helper that only the book website needs. Run it and move on.

In [2]:
# @title Helper for the website versions (run this cell; you do not need to read it)
def slider_figure(make_figure, name, values, start, label=None):
    """Precompute make_figure(name=value) for every value and join the results with one slider.

    The website cannot run Python, so each model figure gets this precomputed version.
    """
    figures = [make_figure(**{name: v}) for v in values]
    first = values.index(start)
    fig = go.Figure(layout=figures[first].layout)
    owner = []                                  # which figure each trace comes from
    for i, f in enumerate(figures):
        for trace in f.data:
            trace.visible = (i == first)
            fig.add_trace(trace)
            owner.append(i)
    steps = []
    for i, f in enumerate(figures):
        layout = {"title": f.layout.title.to_plotly_json(),
                  "annotations": [a.to_plotly_json() for a in f.layout.annotations],
                  "shapes": [s.to_plotly_json() for s in f.layout.shapes]}
        visible = [o == i for o in owner]
        steps.append(dict(method="update", label=str(values[i]), args=[{"visible": visible}, layout]))
    fig.update_layout(sliders=[dict(active=first, steps=steps, pad=dict(t=60),
                                    currentvalue=dict(prefix=(label or name) + " = "))])
    return fig

## The model in code

A full DSGE model has markets for labor, capital and output, and people who plan how much to consume and work
in every future quarter. That forward-looking planning is what makes these models hard to solve. The textbook
keeps the most useful piece, the **labor market**, and treats consumption $c$ as given: the permanent-income
hypothesis says consumption moves little over the business cycle, except after shocks that change people's
lifetime income.

**Labor demand.** Firms produce with the production function of Chapter 4,

$$Y = \bar{A}K^{1/3}L^{2/3}, \qquad (15.1)$$

and hire workers until the marginal product of labor equals the real wage $w$. A tax $\tau$ on every dollar of
sales leaves firms only the share $1-\tau$ of that product (with no tax, $\tau = 0$):

$$\text{Labor demand:}\quad w = (1-\tau)\cdot\frac{2}{3}\cdot\frac{\bar{A}K^{1/3}}{L^{1/3}}. \qquad (15.2)$$

**Labor supply.** People work more when the wage is high relative to their consumption:

$$\text{Labor supply:}\quad L^s = \bar{\ell}\cdot\frac{w}{c}.$$

**Equilibrium.** Write labor supply as a wage, $w = cL/\bar{\ell}$, and set it equal to labor demand. A little
algebra gives employment and the wage directly:

$$L^* = \left[(1-\tau)\cdot\frac{2}{3}\cdot\frac{\bar{A}K^{1/3}\,\bar{\ell}}{c}\right]^{3/4}, \qquad w^* = \frac{c\,L^*}{\bar{\ell}}.$$

**One baseline for every diagram.** The textbook draws its diagrams without numbers, and with straight lines.
We draw the curves that the two equations really give, and every diagram starts from one illustrative set of
values: TFP $\bar{A} = 1$, capital $K = 1$, $\bar{\ell} = 0.9$ and consumption $c = 0.6$. They are chosen so that
the baseline is easy to read: employment $L^* = 1$, output $Y = 1$, and the real wage $w^* = 2/3$. Consumption is
then 60 percent of output, close to its U.S. share of GDP, and labor earns $wL/Y = 2/3$ of output, as in Chapter 4.

> **Python note.** `def` creates a *function*: a named recipe that takes inputs (the *arguments*) and gives
> back a result with `return`. Writing `A=A_BAR` gives an argument a *default value*, used whenever we do not
> pass one. `**` is a power: `L ** (1 / 3)` is the cube root of `L`. A function can return two results at
> once, separated by a comma.

In [3]:
A_BAR, K_BAR, ELL_BAR, C_BAR = 1.0, 1.0, 0.9, 0.6   # the baseline: TFP, capital, labor-supply scale, consumption


def demand_wage(L, A=A_BAR, tau=0.0):
    """Equation 15.2, the labor demand curve: w = (1 - tau) (2/3) A K^(1/3) / L^(1/3)."""
    return (1 - tau) * (2 / 3) * A * K_BAR ** (1 / 3) / L ** (1 / 3)


def supply_wage(L, c=C_BAR, ell=ELL_BAR):
    """The labor supply curve L = ell w / c, written as a wage: w = c L / ell."""
    return c * L / ell


def labor_demand(w, A=A_BAR, tau=0.0):
    """Labor demanded at the real wage w (equation 15.2 solved for L)."""
    return ((1 - tau) * (2 / 3) * A * K_BAR ** (1 / 3) / w) ** 3


def labor_supply(w, c=C_BAR, ell=ELL_BAR):
    """Labor supplied at the real wage w: L = ell w / c."""
    return ell * w / c

In [4]:
def equilibrium(A=A_BAR, tau=0.0, c=C_BAR, ell=ELL_BAR):
    """The labor market clears: employment L* and the real wage w* (formula above)."""
    L = ((1 - tau) * (2 / 3) * A * K_BAR ** (1 / 3) * ell / c) ** 0.75
    return L, supply_wage(L, c, ell)


def output(L, A=A_BAR):
    """Equation 15.1, the production function: Y = A K^(1/3) L^(2/3)."""
    return A * K_BAR ** (1 / 3) * L ** (2 / 3)


def labor_needed(Y, A=A_BAR):
    """Sticky prices: the labor firms need to produce the output Y, L = (Y / (A K^(1/3)))^(3/2)."""
    return (Y / (A * K_BAR ** (1 / 3))) ** 1.5

A quick check of the baseline.

> **Python note.** `L0, w0 = equilibrium()` *unpacks* the two results into two variables. An *f-string* such as
> `f"L* = {L0:.2f}"` inserts a value into text; `:.2f` means "show two decimals".

In [5]:
L0, w0 = equilibrium()
Y_0 = output(L0)
print(f"Employment L* = {L0:.2f},  real wage w* = {w0:.3f},  output Y = {Y_0:.2f}")
print(f"Consumption share c/Y = {C_BAR / Y_0:.2f},  labor share wL/Y = {w0 * L0 / Y_0:.3f}")
print(f"Labor demand and labor supply at w*: {labor_demand(w0):.3f} and {labor_supply(w0):.3f}")

Employment L* = 1.00,  real wage w* = 0.667,  output Y = 1.00
Consumption share c/Y = 0.60,  labor share wL/Y = 0.667
Labor demand and labor supply at w*: 1.000 and 1.000


Both curves pass through the same point, so the market clears at $L^* = 1$ and $w^* = 0.667$.

### Drawing tools

All the labor-market diagrams below use the same axes and the same few drawing steps, so we write them once:

- `add_curve` draws a curve and writes its name at its last point inside the axes;
- `add_point` marks a point with dotted guides to both axes, and `add_arrow` draws an arrow;
- `labor_axes` fixes the axes and writes the title.

> **Python note.** `np.linspace(0.02, 1.8, 179)` creates 179 evenly spaced numbers from 0.02 to 1.8. Because
> our model functions work on a whole array at once, one call gives a whole curve. `go.Figure()` starts an
> empty chart, and `fig.add_trace(go.Scatter(...))` adds one line or set of points to it. `w <= 1.2` is true
> where the wage fits in the chart, `L[inside]` keeps those values, and `[-1]` takes the last one (`[0]` the
> first). `if condition:` runs the indented lines below it only when the condition holds. Labels such as
> `"L<sup>d</sup>"` use HTML to write Lᵈ.

In [6]:
L_GRID = np.linspace(0.02, 1.8, 179)        # employment levels at which we draw the curves
L_RANGE, W_RANGE = [0, 1.8], [0, 1.2]       # fixed axes, so that you can see the curves move


def add_curve(fig, L, w, label, color, dash="solid", avoid=None, start=False):
    """Draw a curve inside the axes and label its last point (its first, top-left point if start),
    unless that point is close to `avoid`."""
    inside = w <= W_RANGE[1]
    L, w = L[inside], w[inside]
    fig.add_trace(go.Scatter(x=L, y=w, mode="lines", name="", showlegend=False,
                             hovertemplate="L = %{x:.2f}, w = %{y:.2f}<extra></extra>",
                             line=dict(color=color, width=3, dash=dash)))
    end = (L[0], w[0]) if start else (L[-1], w[-1])
    if avoid is None or abs(end[0] - avoid[0]) + abs(end[1] - avoid[1]) > 0.1:
        fig.add_annotation(x=end[0], y=end[1], text=label, showarrow=False, yanchor="middle",
                           xanchor="right" if start else "left", font=dict(color=color, size=15))
    return end

In [7]:
def add_point(fig, L, w, label, guides=True, where="top right"):
    """Mark the point (L, w) with a dot and a letter (placed `where`), and (if guides) dotted lines to both axes."""
    if guides:
        fig.add_trace(go.Scatter(x=[0, L, L], y=[w, w, 0], mode="lines", showlegend=False, hoverinfo="skip",
                                 line=dict(color="black", width=1, dash="dot")))
    fig.add_trace(go.Scatter(x=[L], y=[w], mode="markers+text", text=[label], textposition=where,
                             name="", showlegend=False, textfont=dict(size=16), marker=dict(color="black", size=10),
                             hovertemplate="L = %{x:.2f}, w = %{y:.2f}<extra></extra>"))


def add_arrow(fig, x0, y0, x1, y1):
    """Draw an arrow from (x0, y0) to (x1, y1) in axis units; nothing if the two points almost coincide."""
    if abs(x1 - x0) + abs(y1 - y0) > 0.02:
        fig.add_annotation(x=x1, y=y1, ax=x0, ay=y0, xref="x", yref="y", axref="x", ayref="y", text="",
                           showarrow=True, arrowhead=3, arrowwidth=2, arrowcolor="black")


def labor_axes(fig, title, wage="Real wage, w"):
    """Fixed axes for every labor-market diagram (no tick in the bottom-left corner) and the title readout."""
    fig.update_xaxes(title_text="Employment, L", range=L_RANGE, tickvals=[0.5, 1.0, 1.5], zeroline=False)
    fig.update_yaxes(title_text=wage, range=W_RANGE, tickvals=[0.2, 0.4, 0.6, 0.8, 1.0], zeroline=False)
    fig.update_layout(title=title, showlegend=False)
    return fig

## Figure 1: U.S. total factor productivity growth

**Goal.** Look at the shocks that the first DSGE models put at the center of the business cycle.

The first DSGE models, the **real business cycle (RBC)** models of Finn Kydland and Edward Prescott, took the
Solow model of Chapter 5 and let total factor productivity $\bar{A}$ jump up and down from year to year: these are
**TFP shocks**. A good year for TFP brings a boom, a bad year a recession. Are TFP shocks big enough for that?
Growth accounting (Chapter 6) measures TFP growth as the part of output growth that capital and labor do not
explain. The data are John Fernald's series for the U.S. business sector (Federal Reserve Bank of San Francisco).

> **Python note.** `pd.read_csv(...)` reads a data file into a *DataFrame*, a table with named columns.
> `.head()` shows its first five rows.

In [8]:
tfp = pd.read_csv(DATA_URL + "fig15-1.csv")
tfp.head()

,year,tfp_growth
0,1950,2.868240
1,1951,0.335407
2,1952,0.885970
3,1953,4.033540
4,1954,3.855820


Each row is one year: `tfp_growth` is TFP growth in percent. To shade the recessions, we also read the NBER
recession dates, from the month before each recession (the peak) to its last month (the trough).

> **Python note.** `for peak, trough in zip(...)` walks through the two columns together, one recession at a
> time. Our years are plain numbers, so the function turns each date into a decimal year: July 1953 becomes
> 1953 + 6/12 = 1953.5. `fig.add_vrect` shades the band between two x values.

In [9]:
recessions = pd.read_csv(DATA_URL + "recessions.csv", parse_dates=["peak", "trough"])


def shade_recessions(fig):
    """Shade every NBER recession (Figure 15.1), with its dates written as decimal years."""
    for peak, trough in zip(recessions.peak, recessions.trough):
        start = peak.year + (peak.month - 1) / 12
        end = trough.year + trough.month / 12
        fig.add_vrect(x0=start, x1=end, fillcolor=BLUE, opacity=0.12, line_width=0)
    return fig


recessions.tail(3)

,peak,trough
8,2001-03-01,2001-11-01
9,2007-12-01,2009-06-01
10,2020-02-01,2020-04-01


> **Python note.** `px.line(tfp, x="year", y="tfp_growth")` draws a line chart from two columns of the table, and
> `labels={...}` gives the columns readable names on the axes. `tfp.tfp_growth.min()` is the smallest value of the
> column. `fig.update_traces(...)` restyles the line, and `fig.add_hline(y=0)` adds a horizontal line at zero.

In [10]:
low, high = tfp.tfp_growth.min(), tfp.tfp_growth.max()
fig = px.line(tfp, x="year", y="tfp_growth", markers=True,
              labels={"year": "Year", "tfp_growth": "TFP growth (percent per year)"},
              title=f"U.S. TFP growth swings between {low:.1f} and +{high:.1f} percent a year")
fig.update_traces(line=dict(color=BLUE, width=2), marker=dict(size=5))
fig.add_hline(y=0, line_dash="dash", line_color=GRAY)
shade_recessions(fig)
fig.update_xaxes(range=[1948, 2027])
fig.update_yaxes(range=[-3, 5], tickvals=[-2, -1, 0, 1, 2, 3, 4])
fig.show()

**What to notice.**

- TFP growth is far from constant: it ranges from −2.4 percent (1993) to +4.0 percent (1953), and it is negative
  in 16 of the 73 years from 1950 to 2022. Shocks to TFP are real and large.
- But they do not line up with the shaded recessions. In 2001 and 2020 TFP grew by 2.3 and 1.8 percent, and in
  2009, the worst year of the Great Recession, by 3.5 percent, one of the largest values in the series.
- So TFP shocks cannot be the whole story of business cycles. That is one reason DSGE models added many other
  shocks (monetary policy, government purchases, financial frictions) and frictions such as sticky prices.

### Which TFP? The raw Solow residual and the utilization-adjusted series

The textbook caption says the series comes "from the Solow growth accounting formula". That is not quite
right, and the difference matters. In a recession, firms keep some idle workers and machines, or use them less
intensively. Plain growth accounting does not see this, so the slack shows up as a *fall in TFP*. Fernald
estimates this change in **utilization** and removes it; the textbook's line is the *utilization-adjusted*
series. The wider file `tfp_annual.csv` holds both, with the growth of business output:

In [11]:
annual = pd.read_csv(DATA_URL + "tfp_annual.csv")
annual[annual.year.isin([2001, 2008, 2009, 2020])]

,year,gdp_growth,tfp_solow,utilization,tfp_util
51,2001,0.7128,-0.1348,-2.4831,2.3491
58,2008,-1.7041,-2.0614,-1.3409,-0.7189
59,2009,-3.7508,-0.1489,-3.6778,3.5271
70,2020,-2.9012,-0.2302,-2.0451,1.8088


In 2009 output fell by 3.75 percent. The raw Solow residual (`tfp_solow`) fell slightly, −0.15 percent, but
utilization fell by 3.68 percent, so the adjusted TFP growth (`tfp_util`) is +3.53 percent. To compare the two
measures in one chart, we stack them in one long table.

> **Python note.** `annual.year.isin([...])` is true for the years in the list, so the line above keeps four
> rows. `melt` turns columns into rows: the two TFP columns become one column `tfp_growth`, and a new
> column `measure` says which series each row comes from. `.map({...})` replaces each code with a readable
> name. `np.where(condition, a, b)` gives `a` where the condition holds and `b` elsewhere.

In [12]:
long = annual.melt(id_vars=["year", "gdp_growth"], value_vars=["tfp_solow", "tfp_util"],
                   var_name="measure", value_name="tfp_growth")
long["measure"] = long.measure.map({"tfp_solow": "Raw Solow residual",
                                    "tfp_util": "Utilization-adjusted TFP (textbook)"})
long["GDP"] = np.where(long.gdp_growth < 0, "GDP fell", "GDP grew")
long.head()

,year,gdp_growth,measure,tfp_growth,GDP
0,1950,9.4019,Raw Solow residual,7.1698,GDP grew
1,1951,5.8361,Raw Solow residual,2.2484,GDP grew
2,1952,3.3189,Raw Solow residual,1.8537,GDP grew
3,1953,4.7295,Raw Solow residual,2.4164,GDP grew
4,1954,-1.0768,Raw Solow residual,-0.1047,GDP fell


> **Python note.** `facet_col="measure"` draws one panel per measure. `trendline="ols"` adds a least-squares
> regression line, and `trendline_scope="overall"` fits one line per panel through all the points. `.corr()`
> computes the correlation of two columns, between −1 and 1. The loop at the end shortens the panel titles,
> which Plotly writes as "measure=...", and `legend_title_text=""` removes the legend's title.

In [13]:
r_raw = annual.gdp_growth.corr(annual.tfp_solow)
r_util = annual.gdp_growth.corr(annual.tfp_util)
fig = px.scatter(long, x="gdp_growth", y="tfp_growth", facet_col="measure", color="GDP", hover_name="year",
                 trendline="ols", trendline_scope="overall", trendline_color_override=GRAY,
                 color_discrete_map={"GDP grew": BLUE, "GDP fell": GREEN},
                 labels={"gdp_growth": "GDP growth (%)", "tfp_growth": "TFP growth (%)"},
                 range_x=[-4.5, 10], range_y=[-5, 7.5],
                 title=f"Raw TFP moves with GDP (r = {r_raw:.2f}); adjusted TFP does not (r = {r_util:.2f})")
for annotation in fig.layout.annotations:
    annotation.text = annotation.text.split("=")[-1]
fig.update_xaxes(tickvals=[-2, 0, 2, 4, 6, 8])          # no tick labels in the bottom-left corner
fig.update_yaxes(tickvals=[-4, -2, 0, 2, 4, 6])
fig.update_layout(legend_title_text="")
fig.show()

**What to notice.** The raw Solow residual rises and falls with GDP: this strong co-movement was the original
evidence for RBC models. After the utilization adjustment the correlation is about zero. Measured TFP falls in
recessions partly *because* firms use their workers and machines less, not the other way around.

### Key numbers: TFP in recessions

The textbook says TFP growth is negative in some years, but not in 2001 or 2020, and that 2009 brought a big
positive shock. We check these statements, and count how often each TFP measure fell in a year when GDP fell.

> **Python note.** `tfp[tfp.year <= 2022]` keeps the rows for which the condition is true (a *filter*).
> `len(...)` counts rows, and `.sum()` of a true/false column counts the true values.

In [14]:
book_years = tfp[tfp.year <= 2022]                      # the textbook's span
negative = book_years[book_years.tfp_growth < 0]
print(f"Years with negative TFP growth, 1950-2022: {len(negative)}")
print(list(negative.year))
fell = annual[annual.gdp_growth < 0]
print(f"Years in which business output fell: {len(fell)}")
print(f"  raw Solow residual also fell in {(fell.tfp_solow < 0).sum()} of them")
print(f"  utilization-adjusted TFP fell in {(fell.tfp_util < 0).sum()} of them: {list(fell.year[fell.tfp_util < 0])}")

Years with negative TFP growth, 1950-2022: 16
[1959, 1969, 1974, 1979, 1983, 1987, 1993, 1994, 2006, 2007, 2008, 2010, 2011, 2013, 2016, 2022]
Years in which business output fell: 11
  raw Solow residual also fell in 9 of them
  utilization-adjusted TFP fell in 2 of them: [1974, 2008]


These are current data. Fernald revises his series as the BEA and BLS revise theirs, so recent years differ from
the textbook's vintage: 2021 is 1.26 percent here, about 0.45 percent in the textbook's figure.

**Your turn.** Find the five years with the *lowest* utilization-adjusted TFP growth. Were any of them recession
years? Then do the same for the raw Solow residual with `annual`.

<details><summary>Hint</summary>

`tfp.nsmallest(5, "tfp_growth")` gives the five rows with the smallest values. Compare their years with
`recessions`; for the raw residual use `annual.nsmallest(5, "tfp_solow")`.

</details>

In [15]:
# Your turn: try it here

## Figure 2: The labor market in the DSGE model

**Goal.** Draw the labor market of the DSGE model and find its equilibrium.

The labor demand curve slopes down because the marginal product of labor falls as firms hire more workers.
The labor supply curve slopes up because people work more when the wage is higher. They cross at employment
$L_0$ and the real wage $w_0$:

$$L^d:\ w = \frac{2}{3}\cdot\frac{\bar{A}K^{1/3}}{L^{1/3}} \qquad\qquad L^s = \bar{\ell}\cdot\frac{w}{c}$$

The parameter $\bar{\ell}$ sets the overall amount of labor supply: when $c = w$, people supply exactly
$L^s = \bar{\ell}$. Move the sliders to see which curve each parameter shifts.

In [16]:
def labor_market(A=A_BAR, c=C_BAR, ell=ELL_BAR):
    """Figure 15.2: labor demand and labor supply cross at employment L0 and the real wage w0."""
    L0, w0 = equilibrium(A, 0.0, c, ell)
    fig = go.Figure()
    d_end = add_curve(fig, L_GRID, demand_wage(L_GRID, A), "L<sup>d</sup>", BLUE)
    add_curve(fig, L_GRID, supply_wage(L_GRID, c, ell), "L<sup>s</sup>", TEAL, avoid=d_end)
    add_point(fig, L0, w0, "")
    title = f"Equilibrium: employment L₀ = {L0:.2f}, real wage w₀ = {w0:.2f}, output Y = {output(L0, A):.2f}"
    return labor_axes(fig, title)

> **Python note.** `interact` turns each argument of a function into a slider and calls the function again
> every time you move one. The `;` at the end only hides an extra line of text.

In [17]:
interact(labor_market,
         A=FloatSlider(value=1.0, min=0.6, max=1.3, step=0.05, description="TFP Ā", style=WIDE),
         c=FloatSlider(value=0.6, min=0.5, max=0.8, step=0.05, description="consumption c", style=WIDE),
         ell=FloatSlider(value=0.9, min=0.5, max=1.2, step=0.1, description="labor-supply scale ℓ̄", style=WIDE));

In [18]:
# @title Website version: Figure 2 with a precomputed slider for consumption
slider_figure(labor_market, "c", [0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8], start=0.6, label="consumption c")

**What to notice.**

- In the baseline the market clears at $L_0 = 1.00$ and $w_0 = 0.67$.
- $\bar{A}$ appears only in labor demand, so it shifts only the demand curve. Consumption $c$ and $\bar{\ell}$
  appear only in labor supply. Higher consumption shifts labor supply *up and to the left*: richer people want
  more leisure, so they work less at any wage.
- Labor demand is curved and quite flat at the right: a 1 percent lower wage raises the labor firms want by
  3 percent (in $L = (\tfrac{2}{3}\bar{A}K^{1/3}/w)^3$ the wage has a power of 3).

**Your turn.** Raise $\bar{\ell}$ from 0.9 to 1.2. Which curve shifts, and in which direction? What happens to
employment and the real wage? Check your answer with `equilibrium(ell=1.2)`.

<details><summary>Hint</summary>

$\bar{\ell}$ enters $L^s = \bar{\ell}\,w/c$ in the same way as $1/c$, so raising $\bar{\ell}$ has the same effect
as lowering consumption.

</details>

In [19]:
# Your turn: try it here

## Figure 3: A negative TFP shock in the DSGE model

**Goal.** See why a fall in TFP looks like a recession in the labor market.

A temporary fall in TFP from $\bar{A}$ to $\bar{A}'$ lowers the marginal product of labor at every level of
employment, so labor demand shifts **down**. Labor supply does not contain $\bar{A}$, and because the shock is
temporary it barely changes consumption, so the supply curve stays put. The economy moves from $A$ to $B$.

This figure function will serve four textbook figures: a TFP shock (here), a sales tax (Figure 4), government
purchases (Figure 5) and good news about the future (Figure 9). Its three arguments are the three things a shock
can change: TFP $\bar{A}'$, the sales tax $\tau$ and consumption $c'$. When a curve moves, its old position stays
behind as a dashed gray line.

> **Python note.** `abs(x)` is the absolute value of `x`. Slider values are not always exact (0.6 can arrive as
> 0.6000000000000001), so we ask whether a value differs from the baseline by more than 0.001 rather than
> testing `!=`. `a if condition else b` picks `a` when the condition holds and `b` otherwise. `min(a, b)` is the
> smaller of two numbers.

In [20]:
def labor_shock(A_new=A_BAR, tau=0.0, c_new=C_BAR):
    """Figures 15.3-15.5 and 15.14: a shock to TFP, a sales tax or consumption moves the economy from A to B."""
    L0, w0 = equilibrium()
    L1, w1 = equilibrium(A_new, tau, c_new)
    demand_moves = abs(A_new - A_BAR) > 0.001 or tau > 0.001
    supply_moves = abs(c_new - C_BAR) > 0.001
    d_label = "L<sup>d</sup>′" if demand_moves else "L<sup>d</sup>"     # a prime marks a curve that moved
    s_label = "L<sup>s</sup>′" if supply_moves else "L<sup>s</sup>"
    fig = go.Figure()
    d_end = add_curve(fig, L_GRID, demand_wage(L_GRID, A_new, tau), d_label, BLUE)
    s_end = add_curve(fig, L_GRID, supply_wage(L_GRID, c_new), s_label, TEAL, avoid=d_end)
    if demand_moves:
        add_curve(fig, L_GRID, demand_wage(L_GRID), "L<sup>d</sup>", GRAY, "dash", avoid=d_end)
        add_arrow(fig, 1.45, demand_wage(1.45), 1.45, demand_wage(1.45, A_new, tau))
    if supply_moves:
        add_curve(fig, L_GRID, supply_wage(L_GRID), "L<sup>s</sup>", GRAY, "dash", avoid=s_end)
        w_arrow = min(0.95, 1.75 * c_new / ELL_BAR)     # low enough that the arrow ends inside the axes
        add_arrow(fig, labor_supply(w_arrow), w_arrow, labor_supply(w_arrow, c_new), w_arrow)
    add_point(fig, L0, w0, "A")
    if demand_moves or supply_moves:
        add_point(fig, L1, w1, "B")
        add_arrow(fig, 0.05, w0, 0.05, w1)          # the real wage moves ...
        add_arrow(fig, L0, 0.05, L1, 0.05)          # ... and so does employment
    return labor_axes(fig, f"Employment {L0:.2f} → {L1:.2f},   real wage {w0:.2f} → {w1:.2f}")

In [21]:
interact(labor_shock,
         A_new=FloatSlider(value=0.8, min=0.6, max=1.4, step=0.05, description="TFP after the shock Ā′", style=WIDE),
         tau=FloatSlider(value=0.0, min=0.0, max=0.5, step=0.05, description="sales tax τ", style=WIDE),
         c_new=FloatSlider(value=0.6, min=0.4, max=0.8, step=0.05, description="consumption c′", style=WIDE));

In [22]:
# @title Website version: Figure 3 with a precomputed slider for TFP
slider_figure(labor_shock, "A_new", [0.7, 0.8, 0.9, 1.0, 1.1, 1.2], start=0.8, label="TFP after the shock Ā′")

**What to notice.**

- With $\bar{A}' = 0.8$, employment falls from 1.00 to 0.85 and the real wage from 0.67 to 0.56. Both fall, just
  as in a recession.
- Both fall by the same 15 percent. In the formula for $L^*$, TFP has the power $3/4$, so $0.8^{3/4} = 0.85$; and
  the wage is proportional to employment along the unchanged supply curve.
- Move $\bar{A}'$ above 1: a positive TFP shock raises employment *and* the wage, like a boom. This is why the
  first DSGE models focused on TFP shocks.

**Your turn.** How large a fall in TFP cuts employment by exactly 10 percent? Find it with the slider, then
compute it with the formula.

<details><summary>Hint</summary>

You need $(\bar{A}')^{3/4} = 0.9$. Raise both sides to the power $4/3$: try `0.9 ** (4 / 3)`, then check with
`equilibrium(A=...)`.

</details>

In [23]:
# Your turn: try it here

### Textbook Figure 15.11: a TFP shock in an estimated DSGE model

Textbook Figure 15.11 is not recomputed in this notebook. It shows how GDP, consumption, hours worked and
inflation respond, quarter by quarter for five years, to a 1 percent rise in TFP in the estimated model of Frank
Smets and Rafael Wouters, "Shocks and Frictions in U.S. Business Cycles: A Bayesian DSGE Approach," *American
Economic Review* 97(3), 2007, pp. 586–606 ([doi:10.1257/aer.97.3.586](https://doi.org/10.1257/aer.97.3.586)).
Its curves are the output of a large model, with sticky prices and wages, many shocks and dozens of parameters
estimated from U.S. data, solved with specialized software. They are not data that we could download, and reading
numbers off the printed curves would invent a precision that we do not have. So we only report what the textbook
says: GDP rises by more than 1 percent at its peak, because capital accumulates as in the Solow model;
consumption rises by much less, because people smooth it; hours worked *fall* at first, because with sticky
prices firms need fewer workers to meet a demand that rises only slowly; and inflation falls.

The one number we can check is the persistence of the shock. The textbook says that after 4 years (16 quarters)
the TFP shock has decayed to about half its size. If a shock keeps the fraction $\rho$ of its size each quarter,
then $\rho^{16} = 1/2$:

In [24]:
rho = 0.5 ** (1 / 16)
print(f"Quarterly persistence rho = {rho:.3f}: the shock keeps {100 * rho:.1f}% of its size each quarter")
print(f"Left after 1 year: {100 * rho ** 4:.0f}%,  after 4 years: {100 * rho ** 16:.0f}%,  after 8 years: {100 * rho ** 32:.0f}%")

Quarterly persistence rho = 0.958: the shock keeps 95.8% of its size each quarter
Left after 1 year: 84%,  after 4 years: 50%,  after 8 years: 25%


## Figure 4: The imposition of a sales tax in the DSGE model

**Goal.** See that a tax on firms' sales works exactly like a fall in TFP.

With a sales tax $\tau$, firms keep only $1-\tau$ of each dollar of sales, so they hire until the *after-tax*
marginal product equals the wage:

$$w = (1-\tau)\cdot\frac{2}{3}\cdot\frac{\bar{A}K^{1/3}}{L^{1/3}}.$$

The tax shifts labor demand down, and labor supply stays put. We use the same function as in Figure 3, now with
the tax slider set to 20 percent and TFP at its baseline.

In [25]:
interact(labor_shock,
         A_new=FloatSlider(value=1.0, min=0.6, max=1.4, step=0.05, description="TFP Ā′", style=WIDE),
         tau=FloatSlider(value=0.2, min=0.0, max=0.5, step=0.05, description="sales tax τ", style=WIDE),
         c_new=FloatSlider(value=0.6, min=0.4, max=0.8, step=0.05, description="consumption c′", style=WIDE));

In [26]:
# @title Website version: Figure 4 with a precomputed slider for the sales tax
slider_figure(labor_shock, "tau", [0.0, 0.1, 0.2, 0.3, 0.4], start=0.2, label="sales tax τ")

**What to notice.**

- A 20 percent sales tax moves the economy to exactly the same point as a fall in TFP to 0.8 in Figure 3:
  employment 0.85 and a real wage of 0.56. Both lower the term $(1-\tau)\bar{A}$ from 1 to 0.8.
- So a recession-like fall in employment and wages does not require that people "forget" technology. A policy
  change, or another distortion that acts like a tax, can play the role of a negative TFP shock.

### Key numbers: a tax and a TFP shock

In [27]:
L_tax, w_tax = equilibrium(tau=0.2)
L_tfp, w_tfp = equilibrium(A=0.8)
print(f"Sales tax 20%:   employment {L_tax:.4f},  real wage {w_tax:.4f}")
print(f"TFP falls to 0.8: employment {L_tfp:.4f},  real wage {w_tfp:.4f}")

Sales tax 20%:   employment 0.8459,  real wage 0.5639
TFP falls to 0.8: employment 0.8459,  real wage 0.5639


**Your turn.** Find the TFP level $\bar{A}'$ that gives the same equilibrium as a 30 percent sales tax. Check it
by comparing two calls to `equilibrium`.

<details><summary>Hint</summary>

Only the product $(1-\tau)\bar{A}$ matters. Compare `equilibrium(tau=0.3)` with `equilibrium(A=...)`.

</details>

In [28]:
# Your turn: try it here

## Figure 5: A rise in government purchases

**Goal.** See how expectations about future taxes reach today's labor market through consumption.

Government purchases appear in neither labor curve. But a rise in purchases must be paid for with taxes at some
point, so households are poorer over their lifetimes, and by the permanent-income hypothesis they consume less
**today**. Consumption is the variable that lets future events affect the economy now. In $L^s = \bar{\ell}w/c$,
a lower $c$ raises labor supply at every wage: people are poorer, so they also "buy" less leisure. Labor supply
shifts **down and to the right**. Here consumption falls from 0.6 to 0.5, which corresponds to a permanent rise
in purchases of 0.1 (10 percent of output); a temporary rise would cut consumption by less.

In [29]:
interact(labor_shock,
         A_new=FloatSlider(value=1.0, min=0.6, max=1.4, step=0.05, description="TFP Ā′", style=WIDE),
         tau=FloatSlider(value=0.0, min=0.0, max=0.5, step=0.05, description="sales tax τ", style=WIDE),
         c_new=FloatSlider(value=0.5, min=0.4, max=0.8, step=0.05, description="consumption c′", style=WIDE));

In [30]:
# @title Website version: Figure 5 with a precomputed slider for consumption
slider_figure(labor_shock, "c_new", [0.4, 0.45, 0.5, 0.55, 0.6], start=0.5, label="consumption c′")

**What to notice.**

- Employment rises from 1.00 to 1.15, but the real wage *falls*, from 0.67 to 0.64, and so does consumption.
  With capital fixed, the extra workers run into diminishing returns.
- The wage falls much less than employment rises, because labor demand is flat (Figure 2).
- A boom in employment with falling wages and consumption does not look like a normal business cycle. Shocks to
  government purchases are part of the story, but they cannot be its main driver.

**Your turn.** Compare a permanent rise in purchases, which cuts consumption to $c' = 0.5$, with a temporary one
that cuts it only to $c' = 0.58$. How much of the rise in employment is left in the temporary case?

<details><summary>Hint</summary>

Use `equilibrium(c=0.5)` and `equilibrium(c=0.58)`, and compare the rise in employment above 1 in each case.

</details>

In [31]:
# Your turn: try it here

### Textbook Figure 15.12: government purchases in the Smets–Wouters model

Textbook Figure 15.12 is not recomputed in this notebook. It shows the responses of GDP, consumption, hours
worked and inflation over five years to a temporary 1 percent rise in government purchases that decays slowly,
in the Smets–Wouters (2007) model ([doi:10.1257/aer.97.3.586](https://doi.org/10.1257/aer.97.3.586)). Like all
the impulse responses of the chapter, its curves are the solution of a large estimated model, not data, so we do
not redraw them by eye. The textbook reads them as the labor market above predicts: hours worked and GDP rise,
consumption falls, and inflation barely moves. GDP rises by about 1 percent in the short run, a **multiplier**
of about 1, and the effect fades as the extra spending ends.

## Figure 6: Sticky wages in the DSGE model

**Goal.** See how a nominal wage that does not fall creates involuntary unemployment.

Monetary policy needs a distinction between nominal and real variables. Let $W$ be the nominal wage (dollars per
hour) and $P$ the price level, so that $w = W/P$ is the real wage. Suppose the nominal wage is **sticky**:
it is stuck at $W = \bar{W}$. With the price level at $P_0 = 1$ and $\bar{W} = 0.75$, the real wage is 0.75, above
the market-clearing wage of 0.67.

At that wage, people want to supply more labor than firms want to hire. Trade needs both sides, so the short
side of the market decides: firms hire along their demand curve, at point $A$. The gap between the two curves is
**involuntary unemployment**: people who would like to work at the going wage but cannot find a job. The unlabelled
dot marks where the market would clear.

This figure function covers one event in three periods, and Figure 7 reuses it:

- period 0: the nominal wage is stuck at $\bar{W}$ (textbook Figure 15.6);
- period 1: a monetary expansion raises the price level to $P'$, and the real wage falls to $\bar{W}/P'$ (Figure 15.7);
- period 2: eventually the nominal wage adjusts and the market clears again (this last step is not in the textbook
  figures; it is what happens once wages are flexible).

> **Python note.** `real_wages[t]` takes element number `t` of a list, counting from 0, and `"ABC"[t]` the
> letter number `t` of a text. `min(a, b)` is the smaller of two numbers and `max(a, b)` the larger. An
> `IntSlider` is a slider for whole numbers, here the period. A function may have several `return` lines: it stops
> at the first one it reaches.

In [32]:
P0 = 1.0          # the price level before the monetary expansion


def add_wage_line(fig, w, label, color, dash="solid", above=False):
    """A horizontal line at the real wage w, labelled just right of the axes (above or below the line)."""
    fig.add_trace(go.Scatter(x=[0.02, L_RANGE[1]], y=[w, w], mode="lines", showlegend=False, hoverinfo="skip",
                             line=dict(color=color, width=3, dash=dash)))
    fig.add_annotation(x=L_RANGE[1], y=w, text=label, showarrow=False, xanchor="left",
                       yanchor="bottom" if above else "top", font=dict(color=color, size=15))


def add_gap(fig, L_from, L_to, w, text):
    """A bracket from L_from to L_to above the real wage w (clear of the point letters), with a text over it."""
    fig.add_trace(go.Scatter(x=[L_from, L_from, L_to, L_to], y=[w + 0.07, w + 0.1, w + 0.1, w + 0.07],
                             mode="lines", showlegend=False, hoverinfo="skip", line=dict(color="black", width=1.5)))
    fig.add_annotation(x=(L_from + L_to) / 2, y=w + 0.1, text=text, showarrow=False, yanchor="bottom",
                       font=dict(size=13))

In [33]:
def letter_side(w, w_old, w_star):
    """Where to write a point's letter, so that it stays clear of the old wage line w_old above it."""
    if w > w_old - 0.01:
        return "top right"             # no old line just above the point
    if w > w_star + 0.01:
        return "bottom left"           # the point lies on labor demand
    return "bottom right"              # the point lies on labor supply

In [34]:
def sticky_wages(t=0, P_new=1.05, W_bar=0.75):
    """Figures 15.6-15.7: period 0 sticky wage W_bar; period 1 prices rise to P_new; period 2 W has adjusted."""
    L_star, w_star = equilibrium()
    real_wages = [W_bar / P0, W_bar / P_new, w_star]          # W/P in periods 0, 1 and 2
    labels = ["W̄/P<sub>0</sub>", "W̄/P′", "W′/P′"]
    w = real_wages[t]
    employed = min(labor_demand(w), labor_supply(w))          # the short side of the market decides
    gap = max(labor_supply(w) - labor_demand(w), 0.0)
    fig = go.Figure()
    add_curve(fig, L_GRID, demand_wage(L_GRID), "L<sup>d</sup>", BLUE, start=True)   # top label, clear of W/P
    add_curve(fig, L_GRID, supply_wage(L_GRID), "L<sup>s</sup>", TEAL)
    add_point(fig, L_star, w_star, "", guides=False)          # where the market would clear
    if t > 0:
        w_A = real_wages[0]
        L_A = min(labor_demand(w_A), labor_supply(w_A))
        add_wage_line(fig, w_A, labels[0], GRAY, "dash", above=w_A >= w)    # labels on the outer sides
        add_point(fig, L_A, w_A, "A")
        add_arrow(fig, 0.05, w_A, 0.05, w)
        add_arrow(fig, L_A, 0.05, employed, 0.05)
    add_wage_line(fig, w, labels[t], GREEN, above=t > 0 and real_wages[0] < w)
    add_point(fig, employed, w, "ABC"[t], where=letter_side(w, real_wages[0], w_star))
    if gap > 0.02:
        add_gap(fig, labor_demand(w), labor_supply(w), w, "Unemployment")
    return labor_axes(fig, f"Period {t}: real wage W/P = {w:.2f}, employment {employed:.2f}, "
                           f"unemployment {gap:.2f}", "Real wage, W/P")

In [35]:
interact(sticky_wages,
         t=IntSlider(value=0, min=0, max=2, description="period t", style=WIDE),
         P_new=FloatSlider(value=1.05, min=1.0, max=1.25, step=0.05, description="new price level P′", style=WIDE),
         W_bar=FloatSlider(value=0.75, min=0.6, max=0.9, step=0.05, description="sticky nominal wage W̄", style=WIDE));

In [36]:
# @title Website version: Figure 6 with a precomputed slider for the sticky wage
slider_figure(sticky_wages, "W_bar", [0.7, 0.75, 0.8, 0.85, 0.9], start=0.75, label="sticky nominal wage W̄")

**What to notice.**

- At the sticky real wage of 0.75, firms hire 0.70, while people would like to supply 1.125: unemployment is 0.42.
- The higher $\bar{W}$, the larger the gap. Labor demand reacts strongly to the wage, so even a modest excess of
  the wage over its market-clearing level costs many jobs.
- Set $\bar{W}$ below 0.67 and the picture flips: now labor supply is the short side, firms cannot hire all the
  workers they want, and there is no unemployment.

**Your turn.** Find the sticky nominal wage $\bar{W}$ at which unemployment disappears, with $P_0 = 1$.

<details><summary>Hint</summary>

Unemployment is zero when the real wage $\bar{W}/P_0$ equals the market-clearing wage. Print `equilibrium()` to
see that wage.

</details>

In [37]:
# Your turn: try it here

## Figure 7: A monetary expansion with sticky wages

**Goal.** See how monetary policy can raise employment when nominal wages are sticky.

A monetary expansion, a lower fed funds rate, raises spending. In this example only wages are sticky, so the
other prices rise, from $P_0 = 1$ to $P' = 1.05$. With the nominal wage still at $\bar{W}$, the real wage
$\bar{W}/P'$ falls. At the lower real wage firms want more workers and fewer people want to work, so employment
rises from $A$ to $B$ and unemployment shrinks. This is the same function as in Figure 6, now in period 1. Move
the period slider to 2 to let the nominal wage adjust.

In [38]:
interact(sticky_wages,
         t=IntSlider(value=1, min=0, max=2, description="period t", style=WIDE),
         P_new=FloatSlider(value=1.05, min=1.0, max=1.25, step=0.05, description="new price level P′", style=WIDE),
         W_bar=FloatSlider(value=0.75, min=0.6, max=0.9, step=0.05, description="sticky nominal wage W̄", style=WIDE));

In [39]:
# @title Website version: Figure 7 with a precomputed slider for the period
slider_figure(sticky_wages, "t", [0, 1, 2], start=1, label="period t")

**What to notice.**

- A 5 percent rise in prices lowers the real wage from 0.75 to 0.71 and raises employment from 0.70 to 0.81;
  unemployment falls from 0.42 to 0.26.
- The real wage and employment move in **opposite** directions. In actual business cycles, booms do not usually
  come with low real wages, so sticky wages alone cannot be the whole story of how monetary policy works.
- In period 2 the nominal wage has caught up with prices, the real wage is back at 0.67 and the market clears:
  in the long run, money does not change employment (the classical dichotomy).

**Your turn.** Which price level $P'$ removes unemployment completely in period 1? What happens to employment if
prices rise even further?

<details><summary>Hint</summary>

You need $\bar{W}/P' = 2/3$ with $\bar{W} = 0.75$. Above that price level the real wage is below the
market-clearing wage: which side of the market is short now?

</details>

In [40]:
# Your turn: try it here

## Figure 8: A monetary expansion with sticky prices

**Goal.** See why, with sticky prices, labor demand is vertical and a monetary expansion raises both employment
and the real wage.

Now wages are flexible, but firms change their prices only slowly. At its sticky price, each firm agrees to
supply whatever customers demand. So spending decides output $Y$, and the production function decides how much
labor firms must hire to produce it:

$$Y = \bar{A}K^{1/3}L^{2/3} \quad\Longrightarrow\quad L = \left(\frac{Y}{\bar{A}K^{1/3}}\right)^{3/2}.$$

Labor demand no longer depends on the wage: it is a **vertical** line. We start from the flexible-price output
$Y_0 = 1$, so point $A$ is the baseline. A monetary expansion raises the output demanded to $Y' = 1.15$, labor
demand shifts right, and to attract the extra workers the nominal wage must rise. With prices stuck, the real wage
$W/P$ rises too: the economy moves up along the labor supply curve to $B$.

In [41]:
Y0 = 1.0          # output demanded before the shock: the flexible-price output of the baseline


def add_vertical(fig, L, label, color, dash="solid"):
    """A vertical labor demand curve at L (sticky prices), labelled just above the top of the axes."""
    fig.add_trace(go.Scatter(x=[L, L], y=[0.02, W_RANGE[1]], mode="lines", showlegend=False, hoverinfo="skip",
                             line=dict(color=color, width=3, dash=dash)))
    fig.add_annotation(x=L, y=W_RANGE[1], text=label, showarrow=False, yanchor="bottom",
                       font=dict(color=color, size=15))

In [42]:
def sticky_prices(Y_new=1.15, c_new=C_BAR):
    """Figures 15.8 and 15.15: labor demand is the vertical line at the labor needed for the output demanded."""
    L0, L1 = labor_needed(Y0), labor_needed(Y_new)
    w0, w1 = supply_wage(L0), supply_wage(L1, c_new)
    demand_moves, supply_moves = abs(Y_new - Y0) > 0.001, abs(c_new - C_BAR) > 0.001
    d_label = "L<sup>d</sup>′" if demand_moves else "L<sup>d</sup>"
    s_label = "L<sup>s</sup>′" if supply_moves else "L<sup>s</sup>"
    fig = go.Figure()
    add_vertical(fig, L1, d_label, BLUE)
    if demand_moves:
        add_vertical(fig, L0, "L<sup>d</sup>" if abs(L1 - L0) > 0.1 else "", GRAY, "dash")
    end = add_curve(fig, L_GRID, supply_wage(L_GRID, c_new), s_label, TEAL)
    if supply_moves:
        add_curve(fig, L_GRID, supply_wage(L_GRID), "L<sup>s</sup>", GRAY, "dash", avoid=end)
    add_point(fig, L0, w0, "A")
    if demand_moves or supply_moves:
        add_point(fig, L1, w1, "B", where="bottom right")     # below-right of B no curve passes
        add_arrow(fig, L0, 0.05, L1, 0.05)
        add_arrow(fig, 0.05, w0, 0.05, w1)
    title = f"Output {Y0:.2f} → {Y_new:.2f}: employment {L0:.2f} → {L1:.2f}, real wage {w0:.2f} → {w1:.2f}"
    return labor_axes(fig, title, "Real wage, W/P")

In [43]:
interact(sticky_prices,
         Y_new=FloatSlider(value=1.15, min=0.8, max=1.3, step=0.05, description="output demanded Y′", style=WIDE),
         c_new=FloatSlider(value=0.6, min=0.4, max=0.7, step=0.05, description="consumption c′", style=WIDE));

In [44]:
# @title Website version: Figure 8 with a precomputed slider for the output demanded
slider_figure(sticky_prices, "Y_new", [0.9, 1.0, 1.05, 1.1, 1.15, 1.2, 1.3], start=1.15, label="output demanded Y′")

**What to notice.**

- Output demanded rises by 15 percent, employment from 1.00 to 1.23 and the real wage from 0.67 to 0.82.
- Employment and the real wage now move in the **same** direction, as they do over the business cycle. That is
  why models with sticky prices and monetary shocks, the **new Keynesian** models, became so important. Together
  with TFP shocks, they are the two "flavors" that modern DSGE models combine.
- Employment rises more than output (by the power 3/2), because each extra worker adds less output than the last.

**Your turn.** With sticky prices, a rise in TFP can *lower* employment (textbook Exercise 2). Suppose TFP rises by
10 percent while the output demanded stays at $Y = 1$. How much labor do firms need now?

<details><summary>Hint</summary>

`labor_needed` has a second argument: compare `labor_needed(1.0)` with `labor_needed(1.0, A=1.1)`. This is the
fall in hours worked that the textbook finds in Figure 15.11.

</details>

In [45]:
# Your turn: try it here

### Textbook Figures 15.9, 15.10 and 15.13: monetary and financial shocks in the Smets–Wouters model

A full DSGE model reports **impulse response functions**: how one variable, such as GDP, responds over time to
one shock, such as a rise in the fed funds rate. Textbook Figures 15.9, 15.10 and 15.13 show such responses from
the estimated model of Smets and Wouters, "Shocks and Frictions in U.S. Business Cycles: A Bayesian DSGE
Approach," *American Economic Review* 97(3), 2007, pp. 586–606
([doi:10.1257/aer.97.3.586](https://doi.org/10.1257/aer.97.3.586)). They are not recomputed here: they are the
solution of a model with sticky prices and wages, investment adjustment costs and many shocks, whose parameters
were estimated from U.S. data, and they cannot be rebuilt from public data series or read precisely off the page.
What the textbook reports:

- **Figure 15.9**: the response of GDP to a temporary 1 percentage point rise in the fed funds rate. GDP is about
  0.2 percent lower in the first quarter, the peak fall of about 0.33 percent comes 3 to 4 quarters after the shock,
  and most of the effect is gone after 5 years (20 quarters).
- **Figure 15.10**: the same shock, for GDP, consumption, hours worked and inflation. All four fall, as with sticky
  prices in Figure 8 run in reverse. Consumption falls nearly as much as GDP, and inflation is slightly less than
  0.2 percentage points lower for about 2 years.
- **Figure 15.13**: a financial friction shock that raises the interest rate households face by 1 percentage point
  above the fed funds rate. It looks much like a monetary policy shock, with its largest effect on consumption.

## Figure 9: Good news in the DSGE model: flexible prices

**Goal.** See how news about the future changes the labor market today (textbook Exercise 1).

People learn of a discovery that will make the economy more productive in 5 years. Today nothing changes in
production, so labor demand stays put. But people expect to be richer, so by the permanent-income hypothesis they
consume more **today**: consumption rises from 0.6 to $c' = 0.7$. Richer people want more of every good,
including leisure, so labor supply shifts **up and to the left**.

The function `good_news` handles this event twice: with flexible prices it calls `labor_shock` from Figure 3, and
with sticky prices (Figure 10) it calls `sticky_prices` from Figure 8. With sticky prices, firms must also
produce the extra consumption, so the output demanded rises by the same amount as consumption.

> **Python note.** A `ToggleButtons` widget shows a row of buttons, one per option; the function receives the
> text of the selected button.

In [46]:
def good_news(c_new=0.7, prices="flexible"):
    """Figures 15.14-15.15: good news raises consumption today from 0.6 to c_new."""
    if prices == "flexible":
        return labor_shock(c_new=c_new)
    return sticky_prices(Y_new=Y0 + (c_new - C_BAR), c_new=c_new)     # demand rises with consumption

In [47]:
interact(good_news,
         c_new=FloatSlider(value=0.7, min=0.5, max=0.8, step=0.05, description="consumption after the news c′", style=WIDE),
         prices=ToggleButtons(options=["flexible", "sticky"], value="flexible", description="prices"));

In [48]:
# @title Website version: Figure 9 with a precomputed slider for consumption
slider_figure(good_news, "c_new", [0.5, 0.6, 0.7, 0.8], start=0.7, label="consumption after the news c′")

**What to notice.**

- With flexible prices, the good news *lowers* employment today, from 1.00 to 0.89, and raises the real wage a
  little, from 0.67 to 0.69. People work less because they feel richer.
- Nothing about technology has changed yet: the whole effect comes through expectations and consumption, the
  forward-looking variable of the model.

**Your turn.** Textbook Exercise 3 asks about a **permanent** rise in TFP. Then labor demand shifts out *and*
consumption rises. Try a 10 percent rise in both, `labor_shock(A_new=1.1, c_new=0.66)`. What happens to employment
and to the real wage?

<details><summary>Hint</summary>

In the formula for $L^*$, $\bar{A}$ and $c$ enter as the ratio $\bar{A}/c$. What happens to that ratio when both
rise by 10 percent?

</details>

In [49]:
# Your turn: try it here

## Figure 10: Good news in the DSGE model: sticky prices

**Goal.** Compare the same good news with sticky prices (textbook Exercise 1(d)).

With sticky prices, firms supply whatever is demanded. The good news raises consumption, so firms must produce
more and need more workers: the vertical labor demand curve shifts right, from $Y_0 = 1$ to $Y' = 1.1$. At the
same time labor supply still shifts up, because people feel richer. Firms must pay a higher wage to get the
labor they need. Use the buttons to switch between the two cases.

In [50]:
interact(good_news,
         c_new=FloatSlider(value=0.7, min=0.5, max=0.8, step=0.05, description="consumption after the news c′", style=WIDE),
         prices=ToggleButtons(options=["flexible", "sticky"], value="sticky", description="prices"));

In [51]:
# @title Website version: Figure 10 with a precomputed slider between flexible and sticky prices
slider_figure(good_news, "prices", ["flexible", "sticky"], start="sticky", label="prices")

**What to notice.**

- With sticky prices, employment *rises*, from 1.00 to 1.15, and the real wage rises a lot, from 0.67 to 0.90.
  With flexible prices (Figure 9) employment fell to 0.89.
- The same shock moves employment in opposite directions depending on whether prices are sticky. This is why
  macroeconomists argue so much about how sticky prices really are.

**Your turn.** Textbook Exercise 4: government purchases fall for a while. Households expect lower taxes, so
consumption rises to 0.65, but total demand falls, to $Y' = 0.95$. With sticky prices, what happens to employment
and to the real wage? Use `sticky_prices(Y_new=0.95, c_new=0.65)`.

<details><summary>Hint</summary>

The vertical demand curve moves left and labor supply moves up. Employment clearly falls; for the real wage the
two shifts pull in opposite directions. The title rounds both wages to 0.67, so print them with more decimals:
compare `supply_wage(labor_needed(0.95), 0.65)` with `w0`.

</details>

In [52]:
# Your turn: try it here

## Key numbers

These tables collect the main numbers of the chapter. Each one is computed with the functions and data above.
First the labor-market model, all from the illustrative baseline:

In [53]:
L_gov, w_gov = equilibrium(c=0.5)
L_news, w_news = equilibrium(c=0.7)
w_old, w_new = 0.75 / P0, 0.75 / 1.05
rows = [
    ["Baseline: employment L*", L0],
    ["Baseline: real wage w*", w0],
    ["TFP falls to 0.8: employment", L_tfp],
    ["Sales tax of 20%: employment (same as the TFP fall)", L_tax],
    ["Consumption falls to 0.5 (more G): employment", L_gov],
    ["Consumption falls to 0.5 (more G): real wage", w_gov],
    ["Sticky wage 0.75: unemployment", labor_supply(w_old) - labor_demand(w_old)],
    ["Prices rise to 1.05: unemployment", labor_supply(w_new) - labor_demand(w_new)],
    ["Sticky prices, output 1.15: employment", labor_needed(1.15)],
    ["Sticky prices, output 1.15: real wage", supply_wage(labor_needed(1.15))],
    ["Good news, flexible prices: employment", L_news],
    ["Good news, sticky prices: employment", labor_needed(1.1)],
    ["TFP shock with a 16-quarter half-life: quarterly persistence", rho],
]
pd.DataFrame(rows, columns=["Quantity", "Value"]).round(3)

,Quantity,Value
0,Baseline: employment L*,1.000
1,Baseline: real wage w*,0.667
2,TFP falls to 0.8: employment,0.846
3,Sales tax of 20%: employment (same as the TFP ...,0.846
4,Consumption falls to 0.5 (more G): employment,1.147
5,Consumption falls to 0.5 (more G): real wage,0.637
6,Sticky wage 0.75: unemployment,0.423
7,Prices rise to 1.05: unemployment,0.258
8,"Sticky prices, output 1.15: employment",1.233
9,"Sticky prices, output 1.15: real wage",0.822


Then the data of Figure 1.

> **Python note.** `set_index("year")` uses the years as row labels, so `by_year[2009]` is the value for 2009.

In [54]:
by_year = tfp.set_index("year").tfp_growth
rows = [
    ["TFP growth 2001, %", by_year[2001]],
    ["TFP growth 2009, %", by_year[2009]],
    ["TFP growth 2020, %", by_year[2020]],
    ["Lowest TFP growth (1993), %", by_year.min()],
    ["Highest TFP growth (1953), %", by_year.max()],
    ["Years with negative TFP growth, 1950-2022", len(negative)],
    ["Correlation of GDP growth with the raw Solow residual", r_raw],
    ["Correlation of GDP growth with adjusted TFP growth", r_util],
]
pd.DataFrame(rows, columns=["Quantity", "Value"]).round(2)

,Quantity,Value
0,"TFP growth 2001, %",2.35
1,"TFP growth 2009, %",3.53
2,"TFP growth 2020, %",1.81
3,"Lowest TFP growth (1993), %",-2.42
4,"Highest TFP growth (1953), %",4.03
5,"Years with negative TFP growth, 1950-2022",16.00
6,Correlation of GDP growth with the raw Solow r...,0.82
7,Correlation of GDP growth with adjusted TFP gr...,-0.03


## Summary

- **DSGE models** are *dynamic* (they trace the economy over time), *stochastic* (shocks hit it) and *general
  equilibrium* (prices clear the markets). They grew out of **real business cycle** models, which added TFP shocks
  to the Solow model; U.S. TFP growth swings a lot, but once utilization is measured it does not fall in most
  recessions.
- The **labor market** is a key block of every DSGE model. With consumption taken as given, a TFP fall or a sales
  tax lowers employment and the wage together, like a recession; a rise in government purchases or good news acts
  through consumption and labor supply instead.
- **Nominal rigidities** decide how monetary policy works. With sticky wages, an expansion raises employment but
  lowers the real wage; with sticky prices, labor demand is vertical and an expansion raises both.
- Estimated DSGE models such as Smets–Wouters are **quantitative**: their impulse response functions say how much
  and how fast GDP, consumption, hours and inflation respond to each shock.
- The next chapter studies **consumption** itself: the permanent-income hypothesis and the microfoundations behind
  the consumption that this chapter took as given.